# 02d DFSI Expression Audit

This notebook rebuilds the DFSI expression audit from raw CSV columns and does not proceed to sensitivity analysis. It does not modify `data/raw`. The audit intentionally does not reuse prior calculated DFSI columns or products to create the fresh value.

## Setup

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import nbformat

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_colwidth", 220)

EXPECTED_RAW_FILES = [
    "DFSI.csv",
    "District_FloodedArea.csv",
    "District_FloodImpact.csv",
    "India_Flood_Inventory_v3.csv",
]


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        raw_dir = candidate / "data" / "raw"
        if all((raw_dir / file_name).exists() for file_name in EXPECTED_RAW_FILES):
            return candidate
    raise FileNotFoundError("Could not find repository root containing data/raw with expected CSV files.")


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/abhijithshaji/Documents/india-flood-index-robustness


## Reload Raw CSV Files

In [2]:
dfsi_raw = pd.read_csv(RAW_DIR / "DFSI.csv", low_memory=False)
impact_raw = pd.read_csv(RAW_DIR / "District_FloodImpact.csv", low_memory=False)
area_raw = pd.read_csv(RAW_DIR / "District_FloodedArea.csv", low_memory=False)
inventory_raw = pd.read_csv(RAW_DIR / "India_Flood_Inventory_v3.csv", low_memory=False)

for name, table in {
    "DFSI.csv": dfsi_raw,
    "District_FloodImpact.csv": impact_raw,
    "District_FloodedArea.csv": area_raw,
    "India_Flood_Inventory_v3.csv": inventory_raw,
}.items():
    print(f"{name}: {table.shape[0]:,} rows x {table.shape[1]:,} columns")
    print(list(table.columns))

DFSI.csv: 744 rows x 3 columns
['Unnamed: 0', 'State_Name', 'DFSI']
District_FloodImpact.csv: 732 rows x 5 columns
['Dist_Name', 'Human_fatality', 'Human_injured', 'Population', 'Mean_Flood_Duration']
District_FloodedArea.csv: 732 rows x 4 columns
['Dist_Name', 'Percent_Flooded_Area', 'Parmanent_Water', 'Corrected_Percent_Flooded_Area']
India_Flood_Inventory_v3.csv: 6,876 rows x 23 columns
['Unnamed: 0', 'UEI', 'Start Date', 'End Date', 'Duration(Days)', 'Main Cause', 'Location', 'Districts', 'State', 'Latitude', 'Longitude', 'Severity', 'Area Affected', 'Human fatality', 'Human injured', 'Human Displaced', 'Animal Fatality', 'Description of Casualties/injured', 'Extent of damage ', 'Event Source', 'Event Souce ID', 'District_LGD_Codes', 'State_Codes']


## Scan Previous Notebook Source Code

The scan searches prior notebook code cells for the requested expression patterns and stale-column risks.

In [3]:
scan_patterns = [
    {"pattern_name": "-np.log10", "regex": r"-\s*np\.log10"},
    {"pattern_name": "np.reciprocal", "regex": r"np\.reciprocal"},
    {"pattern_name": "1 /", "regex": r"1\s*/"},
    {"pattern_name": "StandardScaler", "regex": r"StandardScaler"},
    {"pattern_name": "MinMaxScaler", "regex": r"MinMaxScaler"},
    {"pattern_name": "subtraction_between_log_terms", "regex": r"log10\([^\n]*\)\s*-|-[^\n]*log10\("},
    {"pattern_name": "assignments_to_reconstructed_dfsi", "regex": r"reconstructed_dfsi\s*="},
    {"pattern_name": "overwriting_of_calculated_dfsi", "regex": r"calculated_dfsi\s*="},
    {"pattern_name": "reuse_of_stale_dataframe_columns", "regex": r"dfsi_calculated|reconstructed_dfsi|dfsi_float64_product|dfsi_product_fresh|product_fresh"},
]

scan_records = []
for notebook_path in sorted(NOTEBOOK_DIR.glob("*.ipynb")):
    if notebook_path.name == "02d_dfsi_expression_audit.ipynb":
        continue
    try:
        nb = nbformat.read(notebook_path, as_version=4)
    except Exception as exc:
        scan_records.append({
            "notebook": notebook_path.name,
            "cell_index": pd.NA,
            "pattern_name": "notebook_read_error",
            "line_number": pd.NA,
            "line_text": str(exc),
        })
        continue

    for cell_index, cell in enumerate(nb.cells):
        if cell.cell_type != "code":
            continue
        for line_number, line_text in enumerate(cell.source.splitlines(), start=1):
            for pattern in scan_patterns:
                if re.search(pattern["regex"], line_text):
                    scan_records.append({
                        "notebook": notebook_path.name,
                        "cell_index": cell_index,
                        "pattern_name": pattern["pattern_name"],
                        "line_number": line_number,
                        "line_text": line_text.strip(),
                    })

dfsi_expression_source_scan = pd.DataFrame(scan_records)
display(dfsi_expression_source_scan)
dfsi_expression_source_scan.to_csv(TABLE_DIR / "dfsi_expression_source_scan.csv", index=False)

,notebook,cell_index,pattern_name,line_number,line_text
0,02_dfsi_reproduction.ipynb,23,reuse_of_stale_dataframe_columns,26,"comparison[""dfsi_float64_product""] = ("
1,02_dfsi_reproduction.ipynb,23,reuse_of_stale_dataframe_columns,36,"assert comparison.loc[complete_component_mask, ""dfsi_float64_product""].gt(0).all(), ""Complete-row product must be positive"""
2,02_dfsi_reproduction.ipynb,23,reuse_of_stale_dataframe_columns,38,"comparison[""dfsi_calculated""] = ("
3,02_dfsi_reproduction.ipynb,23,-np.log10,40,"- np.log10(1.0 + comparison[""component_mean_flood_duration""])"
4,02_dfsi_reproduction.ipynb,23,subtraction_between_log_terms,40,"- np.log10(1.0 + comparison[""component_mean_flood_duration""])"
...,...,...,...,...,...
70,02c_dfsi_target_column_audit.ipynb,15,reuse_of_stale_dataframe_columns,41,"{""metric"": ""spearman_reconstructed_vs_negative_published_dfsi_rank"", ""value"": verified_finite[""reconstructed_dfsi""].corr(-verified_finite[""published_dfsi_score_rank""].astype(""float64""), method=""spearman"")},"
71,02c_dfsi_target_column_audit.ipynb,15,reuse_of_stale_dataframe_columns,42,"{""metric"": ""pearson_reconstructed_vs_published_dfsi_score"", ""value"": verified_finite[""reconstructed_dfsi""].corr(verified_finite[""published_dfsi_score""], method=""pearson"")},"
72,02c_dfsi_target_column_audit.ipynb,15,reuse_of_stale_dataframe_columns,46,"{""metric"": ""reconstructed_dfsi_min"", ""value"": verified_finite[""reconstructed_dfsi""].min()},"
73,02c_dfsi_target_column_audit.ipynb,15,reuse_of_stale_dataframe_columns,47,"{""metric"": ""reconstructed_dfsi_max"", ""value"": verified_finite[""reconstructed_dfsi""].max()},"


## Raw Component Mapping

In [4]:
def make_review_key(value):
    if pd.isna(value):
        return pd.NA
    return re.sub(r"\s+", " ", str(value).strip()).casefold()


def split_list_cell(value):
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split(",")]


def clean_lgd_code(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    if text == "" or text.casefold() in {"none", "nan", "na", "<na>"}:
        return pd.NA
    return text


component_mapping = pd.DataFrame([
    {"symbol": "E", "component": "number of flood events", "raw_source": "India_Flood_Inventory_v3.csv", "raw_column": "UEI counted after exploding Districts", "conversion": "pd.to_numeric(..., errors='coerce').astype('float64')"},
    {"symbol": "D", "component": "mean flood duration", "raw_source": "District_FloodImpact.csv", "raw_column": "Mean_Flood_Duration", "conversion": "pd.to_numeric(..., errors='coerce').astype('float64')"},
    {"symbol": "F", "component": "human fatalities", "raw_source": "District_FloodImpact.csv", "raw_column": "Human_fatality", "conversion": "pd.to_numeric(..., errors='coerce').astype('float64')"},
    {"symbol": "I", "component": "humans injured", "raw_source": "District_FloodImpact.csv", "raw_column": "Human_injured", "conversion": "pd.to_numeric(..., errors='coerce').astype('float64')"},
    {"symbol": "A", "component": "corrected flooded area", "raw_source": "District_FloodedArea.csv", "raw_column": "Corrected_Percent_Flooded_Area", "conversion": "pd.to_numeric(..., errors='coerce').astype('float64')"},
    {"symbol": "P", "component": "population", "raw_source": "District_FloodImpact.csv", "raw_column": "Population", "conversion": "pd.to_numeric(..., errors='coerce').astype('float64')"},
])
display(component_mapping)

,symbol,component,raw_source,raw_column,conversion
0,E,number of flood events,India_Flood_Inventory_v3.csv,UEI counted after exploding Districts,"pd.to_numeric(..., errors='coerce').astype('float64')"
1,D,mean flood duration,District_FloodImpact.csv,Mean_Flood_Duration,"pd.to_numeric(..., errors='coerce').astype('float64')"
2,F,human fatalities,District_FloodImpact.csv,Human_fatality,"pd.to_numeric(..., errors='coerce').astype('float64')"
3,I,humans injured,District_FloodImpact.csv,Human_injured,"pd.to_numeric(..., errors='coerce').astype('float64')"
4,A,corrected flooded area,District_FloodedArea.csv,Corrected_Percent_Flooded_Area,"pd.to_numeric(..., errors='coerce').astype('float64')"
5,P,population,District_FloodImpact.csv,Population,"pd.to_numeric(..., errors='coerce').astype('float64')"


## Fresh One-to-One Join From Raw Sources

Event counts are rebuilt by exploding raw inventory district names and counting distinct `UEI`. The district summary tables and `DFSI.csv` still lack LGD codes, so the one-to-one table excludes repeated district names.

In [5]:
event_rows = []
for _, row in inventory_raw.reset_index(names="inventory_source_row_id").iterrows():
    districts = split_list_cell(row["Districts"])
    district_codes = split_list_cell(row["District_LGD_Codes"])
    for position, district_name in enumerate(districts):
        event_rows.append({
            "inventory_source_row_id": row["inventory_source_row_id"],
            "UEI": row.get("UEI"),
            "district_name_from_inventory": district_name,
            "district_review_key": make_review_key(district_name),
            "district_lgd_code": clean_lgd_code(district_codes[position] if position < len(district_codes) else pd.NA),
        })

event_districts_fresh = pd.DataFrame(event_rows)
event_counts_fresh = (
    event_districts_fresh.groupby("district_review_key", dropna=False)
    .agg(
        raw_E_flood_events=("UEI", "nunique"),
        inventory_district_names=("district_name_from_inventory", lambda s: " | ".join(sorted(set(map(str, s.dropna()))))),
        lgd_codes_seen=("district_lgd_code", lambda s: " | ".join(sorted(set(map(str, s.dropna()))))),
        lgd_code_count=("district_lgd_code", lambda s: int(s.dropna().nunique())),
    )
    .reset_index()
)

impact_fresh = impact_raw.reset_index(names="impact_source_row_id").copy()
impact_fresh["district_review_key"] = impact_fresh["Dist_Name"].map(make_review_key)
area_fresh = area_raw.reset_index(names="area_source_row_id").copy()
area_fresh["district_review_key"] = area_fresh["Dist_Name"].map(make_review_key)
published_fresh = dfsi_raw.reset_index(names="dfsi_source_row_id").rename(columns={"Unnamed: 0": "published_district"}).copy()
published_fresh["district_review_key"] = published_fresh["published_district"].map(make_review_key)
published_fresh["published_dfsi"] = pd.to_numeric(published_fresh["DFSI"], errors="coerce").astype("float64")

impact_counts = impact_fresh["district_review_key"].value_counts(dropna=False).rename("impact_count")
area_counts = area_fresh["district_review_key"].value_counts(dropna=False).rename("area_count")
published_counts = published_fresh["district_review_key"].value_counts(dropna=False).rename("published_count")
key_audit_fresh = pd.concat([impact_counts, area_counts, published_counts], axis=1).fillna(0).astype(int).reset_index().rename(columns={"index": "district_review_key"})
one_to_one_keys = key_audit_fresh.query("impact_count == 1 and area_count == 1 and published_count == 1")["district_review_key"]

fresh_base = (
    impact_fresh[impact_fresh["district_review_key"].isin(one_to_one_keys)]
    .merge(area_fresh[area_fresh["district_review_key"].isin(one_to_one_keys)], on="district_review_key", how="inner", suffixes=("_impact", "_area"), validate="one_to_one")
    .merge(event_counts_fresh, on="district_review_key", how="left", validate="one_to_one")
    .merge(published_fresh[published_fresh["district_review_key"].isin(one_to_one_keys)][["dfsi_source_row_id", "published_district", "State_Name", "published_dfsi", "district_review_key"]], on="district_review_key", how="inner", validate="one_to_one")
)
fresh_base["raw_E_flood_events_missing"] = fresh_base["raw_E_flood_events"].isna()
fresh_base["raw_E_flood_events"] = fresh_base["raw_E_flood_events"].fillna(0)

print(f"Fresh one-to-one rows: {len(fresh_base):,}")

Fresh one-to-one rows: 720


## Fresh Term Contributions

Components are independently converted to float64. The requested term columns and fresh expressions are calculated directly from those converted raw components.

In [6]:
term_contrib = fresh_base.copy()

# Raw components converted independently from the raw source columns.
term_contrib["E"] = pd.to_numeric(term_contrib["raw_E_flood_events"], errors="coerce").astype("float64")
term_contrib["D"] = pd.to_numeric(term_contrib["Mean_Flood_Duration"], errors="coerce").astype("float64")
term_contrib["F"] = pd.to_numeric(term_contrib["Human_fatality"], errors="coerce").astype("float64")
term_contrib["I"] = pd.to_numeric(term_contrib["Human_injured"], errors="coerce").astype("float64")
term_contrib["A"] = pd.to_numeric(term_contrib["Corrected_Percent_Flooded_Area"], errors="coerce").astype("float64")
term_contrib["P"] = pd.to_numeric(term_contrib["Population"], errors="coerce").astype("float64")

raw_component_columns = ["E", "D", "F", "I", "A", "P"]
complete_row_mask = term_contrib[raw_component_columns].notna().all(axis=1)

term_contrib["term_events"] = np.log10(1.0 + term_contrib["E"])
term_contrib["term_duration"] = np.log10(1.0 + term_contrib["D"])
term_contrib["term_fatalities"] = 2.0 * np.log10(1.0 + term_contrib["F"])
term_contrib["term_injured"] = np.log10(1.0 + term_contrib["I"])
term_contrib["term_flooded_area"] = np.log10(1.0 + term_contrib["A"])
term_contrib["term_population"] = np.log10(term_contrib["P"])

log_term_columns = ["term_events", "term_duration", "term_fatalities", "term_injured", "term_flooded_area", "term_population"]

term_contrib["dfsi_fresh"] = (
    term_contrib["term_events"]
    - term_contrib["term_duration"]
    - term_contrib["term_fatalities"]
    - term_contrib["term_injured"]
    - term_contrib["term_flooded_area"]
    - term_contrib["term_population"]
)

term_contrib["product_fresh"] = (
    (1.0 + term_contrib["E"])
    - (1.0 + term_contrib["D"])
    - (1.0 + term_contrib["F"]) ** 2.0
    - (1.0 + term_contrib["I"])
    - (1.0 + term_contrib["A"])
    - term_contrib["P"]
)
term_contrib["dfsi_product_fresh"] = np.where(term_contrib["product_fresh"] > 0, np.log10(term_contrib["product_fresh"]), np.nan)

# Extra diagnostics requested for old-vs-fresh tests.
term_contrib["negative_dfsi_fresh"] = -term_contrib["dfsi_fresh"]
term_contrib["log10_one_over_product_fresh"] = np.where(term_contrib["product_fresh"] > 0, np.log10(1.0 / term_contrib["product_fresh"]), np.nan)
term_contrib["negative_sum_of_six_terms"] = -term_contrib[log_term_columns].sum(axis=1)
term_contrib["subtracted_log_terms_expression"] = term_contrib["dfsi_fresh"]

normalised_terms = pd.DataFrame(index=term_contrib.index)
for column in raw_component_columns:
    max_value = term_contrib[column].max(skipna=True)
    normalised_terms[column] = term_contrib[column] / max_value if pd.notna(max_value) and max_value != 0 else np.nan
term_contrib["normalised_component_log_expression"] = (
    np.log10(1.0 + normalised_terms["E"])
    - np.log10(1.0 + normalised_terms["D"])
    - 2.0 * np.log10(1.0 + normalised_terms["F"])
    - np.log10(1.0 + normalised_terms["I"])
    - np.log10(1.0 + normalised_terms["A"])
    - np.log10(normalised_terms["P"])
)

display(term_contrib[["published_district", "State_Name", *raw_component_columns, *log_term_columns, "dfsi_fresh", "product_fresh", "dfsi_product_fresh", "published_dfsi"]].head())

/Users/abhijithshaji/Documents/india-flood-index-robustness/.venv/lib/python3.9/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/abhijithshaji/Documents/india-flood-index-robustness/.venv/lib/python3.9/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


,published_district,State_Name,E,D,F,I,A,P,term_events,term_duration,term_fatalities,term_injured,term_flooded_area,term_population,dfsi_fresh,product_fresh,dfsi_product_fresh,published_dfsi
0,Dadra & Nagar Haveli,DADRA & NAGAR HAVE,0.0,1.0,10.0,0.0,2.969168,618459.0,0.0,0.301030,2.082785,0.0,0.598699,5.791311,-8.773826,-618585.969168,NaN,9.820211
1,Daman,DAMAN & DIU,0.0,NaN,0.0,0.0,1.544538,190125.0,0.0,NaN,0.000000,0.0,0.405609,5.279039,NaN,NaN,NaN,5.233751
2,Diu,DAMAN & DIU,0.0,NaN,0.0,0.0,0.819836,67412.0,0.0,NaN,0.000000,0.0,0.260032,4.828737,NaN,NaN,NaN,4.269142
3,North & Middle Andaman,ANDAMAN & NICOBAR,0.0,1.0,0.0,0.0,1.314817,101677.0,0.0,0.301030,0.000000,0.0,0.364517,5.007223,-5.672769,-101682.314817,NaN,7.462575
4,South Andaman,ANDAMAN & NICOBAR,0.0,2.0,1.0,0.0,1.443452,257900.0,0.0,0.477121,0.602060,0.0,0.388004,5.411451,-6.878636,-257909.443452,NaN,8.360463


## Previous Versus Fresh Calculation Tests

The previous reconstructed value is read only for comparison; it is not used in the fresh calculation.

In [7]:
previous_paths = [
    TABLE_DIR / "dfsi_verified_comparison.csv",
    TABLE_DIR / "dfsi_corrected_comparison.csv",
    TABLE_DIR / "dfsi_reproduction_comparison.csv",
]
previous_df = None
previous_source = None
for path in previous_paths:
    if path.exists():
        candidate_previous = pd.read_csv(path)
        if {"district_review_key"}.issubset(candidate_previous.columns):
            previous_df = candidate_previous
            previous_source = path.name
            break

if previous_df is None:
    term_contrib["previous_reconstructed_dfsi"] = np.nan
    previous_source = "not_available"
else:
    if "reconstructed_dfsi" in previous_df.columns:
        previous_value_column = "reconstructed_dfsi"
    elif "dfsi_calculated" in previous_df.columns:
        previous_value_column = "dfsi_calculated"
    else:
        previous_value_column = None

    if previous_value_column is None:
        term_contrib["previous_reconstructed_dfsi"] = np.nan
    else:
        previous_lookup = previous_df[["district_review_key", previous_value_column]].rename(columns={previous_value_column: "previous_reconstructed_dfsi"})
        term_contrib = term_contrib.merge(previous_lookup, on="district_review_key", how="left", validate="one_to_one")

old_vs_fresh = term_contrib[[
    "district_review_key", "published_district", "State_Name", "previous_reconstructed_dfsi", "dfsi_fresh", "negative_dfsi_fresh",
    "log10_one_over_product_fresh", "negative_sum_of_six_terms", "subtracted_log_terms_expression", "normalised_component_log_expression", "published_dfsi"
]].copy()

candidate_columns = [
    "negative_dfsi_fresh",
    "log10_one_over_product_fresh",
    "negative_sum_of_six_terms",
    "subtracted_log_terms_expression",
    "normalised_component_log_expression",
]
for column in candidate_columns:
    old_vs_fresh[f"previous_equals_{column}"] = np.isclose(
        old_vs_fresh["previous_reconstructed_dfsi"], old_vs_fresh[column], rtol=1e-10, atol=1e-10, equal_nan=False
    )

old_vs_fresh["previous_source_file"] = previous_source
old_vs_fresh.to_csv(TABLE_DIR / "dfsi_old_vs_fresh_calculation.csv", index=False)
display(old_vs_fresh.head())

,district_review_key,published_district,State_Name,previous_reconstructed_dfsi,dfsi_fresh,negative_dfsi_fresh,log10_one_over_product_fresh,negative_sum_of_six_terms,subtracted_log_terms_expression,normalised_component_log_expression,published_dfsi,previous_equals_negative_dfsi_fresh,previous_equals_log10_one_over_product_fresh,previous_equals_negative_sum_of_six_terms,previous_equals_subtracted_log_terms_expression,previous_equals_normalised_component_log_expression,previous_source_file
0,dadra & nagar haveli,Dadra & Nagar Haveli,DADRA & NAGAR HAVE,-8.773826,-8.773826,8.773826,NaN,-8.773826,-8.773826,1.267250,9.820211,False,False,True,True,False,dfsi_corrected_comparison.csv
1,daman,Daman,DAMAN & DIU,NaN,NaN,NaN,NaN,-5.684648,NaN,NaN,5.233751,False,False,False,False,False,dfsi_corrected_comparison.csv
2,diu,Diu,DAMAN & DIU,NaN,NaN,NaN,NaN,-5.088770,NaN,NaN,4.269142,False,False,False,False,False,dfsi_corrected_comparison.csv
3,north & middle andaman,North & Middle Andaman,ANDAMAN & NICOBAR,-5.672769,-5.672769,5.672769,NaN,-5.672769,-5.672769,2.084253,7.462575,False,False,True,True,False,dfsi_corrected_comparison.csv
4,south andaman,South Andaman,ANDAMAN & NICOBAR,-6.878636,-6.878636,6.878636,NaN,-6.878636,-6.878636,1.665387,8.360463,False,False,True,True,False,dfsi_corrected_comparison.csv


## Assertion Audit and Metrics

The requested assertions are evaluated and recorded. Failed assertions are saved instead of hidden, because several requested assertions are not true for the raw data and expression as written.

In [8]:
def assertion_record(assertion_name, passed, failed_count, details):
    return {
        "audit_type": "assertion",
        "check_name": assertion_name,
        "passed": bool(passed),
        "affected_rows": int(failed_count) if pd.notna(failed_count) else failed_count,
        "details": details,
    }

complete = term_contrib.loc[complete_row_mask].copy()
assertion_records = []
assertion_records.append(assertion_record(
    "all_complete_row_raw_components_are_nonnegative",
    complete[raw_component_columns].ge(0).all().all(),
    int(complete[raw_component_columns].lt(0).any(axis=1).sum()),
    "Checks E, D, F, I, A, and P after independent float64 conversion.",
))
assertion_records.append(assertion_record(
    "population_is_greater_than_zero",
    complete["P"].gt(0).all(),
    int((complete["P"] <= 0).sum()),
    "Required because term_population = log10(P).",
))
assertion_records.append(assertion_record(
    "every_complete_row_log_term_is_nonnegative",
    complete[log_term_columns].ge(0).all().all(),
    int(complete[log_term_columns].lt(0).any(axis=1).sum()),
    "Each term column is expected to be nonnegative for nonnegative components and P > 0; P >= 1 in these data.",
))
assertion_records.append(assertion_record(
    "product_fresh_is_greater_than_zero",
    complete["product_fresh"].gt(0).all(),
    int((complete["product_fresh"] <= 0).sum()),
    "This uses the requested raw-subtraction expression for product_fresh.",
))
assertion_records.append(assertion_record(
    "dfsi_fresh_is_greater_than_zero",
    complete["dfsi_fresh"].gt(0).all(),
    int((complete["dfsi_fresh"] <= 0).sum()),
    "This uses term_events minus all other log terms exactly as requested.",
))
comparison_available = complete[["dfsi_fresh", "dfsi_product_fresh"]].notna().all(axis=1)
if comparison_available.any():
    max_product_diff = (complete.loc[comparison_available, "dfsi_fresh"] - complete.loc[comparison_available, "dfsi_product_fresh"]).abs().max()
    agree = bool(max_product_diff <= 1e-10)
    failed = int(((complete.loc[comparison_available, "dfsi_fresh"] - complete.loc[comparison_available, "dfsi_product_fresh"]).abs() > 1e-10).sum())
else:
    max_product_diff = np.nan
    agree = False
    failed = int(len(complete))
assertion_records.append(assertion_record(
    "dfsi_fresh_and_dfsi_product_fresh_agree_within_1e_10",
    agree,
    failed,
    f"Comparable rows: {int(comparison_available.sum())}; max absolute difference: {max_product_diff}.",
))

finite = term_contrib.replace([np.inf, -np.inf], np.nan).dropna(subset=["dfsi_fresh", "published_dfsi"])
pearson = finite["dfsi_fresh"].corr(finite["published_dfsi"], method="pearson") if len(finite) >= 2 else np.nan
spearman = finite["dfsi_fresh"].corr(finite["published_dfsi"], method="spearman") if len(finite) >= 2 else np.nan
median_abs_diff = (finite["dfsi_fresh"] - finite["published_dfsi"]).abs().median() if len(finite) else np.nan

metric_records = [
    {"audit_type": "metric", "check_name": "one_to_one_rows", "passed": pd.NA, "affected_rows": len(term_contrib), "details": "Rows after fresh raw-source one-to-one district-name join."},
    {"audit_type": "metric", "check_name": "complete_rows", "passed": pd.NA, "affected_rows": len(complete), "details": "Rows with all six raw components present."},
    {"audit_type": "metric", "check_name": "dfsi_fresh_min", "passed": pd.NA, "affected_rows": pd.NA, "details": complete["dfsi_fresh"].min()},
    {"audit_type": "metric", "check_name": "dfsi_fresh_max", "passed": pd.NA, "affected_rows": pd.NA, "details": complete["dfsi_fresh"].max()},
    {"audit_type": "metric", "check_name": "product_fresh_min", "passed": pd.NA, "affected_rows": pd.NA, "details": complete["product_fresh"].min()},
    {"audit_type": "metric", "check_name": "product_fresh_max", "passed": pd.NA, "affected_rows": pd.NA, "details": complete["product_fresh"].max()},
    {"audit_type": "metric", "check_name": "pearson_dfsi_fresh_vs_published_dfsi", "passed": pd.NA, "affected_rows": len(finite), "details": pearson},
    {"audit_type": "metric", "check_name": "spearman_dfsi_fresh_vs_published_dfsi", "passed": pd.NA, "affected_rows": len(finite), "details": spearman},
    {"audit_type": "metric", "check_name": "median_absolute_difference", "passed": pd.NA, "affected_rows": len(finite), "details": median_abs_diff},
    {"audit_type": "metric", "check_name": "previous_source_file", "passed": pd.NA, "affected_rows": pd.NA, "details": previous_source},
    {"audit_type": "old_value_test", "check_name": "previous_equals_negative_dfsi_fresh_count", "passed": pd.NA, "affected_rows": int(old_vs_fresh["previous_equals_negative_dfsi_fresh"].sum()), "details": "Rows where previous reconstructed value equals -dfsi_fresh within 1e-10."},
    {"audit_type": "old_value_test", "check_name": "previous_equals_log10_one_over_product_fresh_count", "passed": pd.NA, "affected_rows": int(old_vs_fresh["previous_equals_log10_one_over_product_fresh"].sum()), "details": "Rows where previous reconstructed value equals log10(1 / product_fresh) within 1e-10."},
    {"audit_type": "old_value_test", "check_name": "previous_equals_negative_sum_of_six_terms_count", "passed": pd.NA, "affected_rows": int(old_vs_fresh["previous_equals_negative_sum_of_six_terms"].sum()), "details": "Rows where previous reconstructed value equals negative sum of the six terms within 1e-10."},
    {"audit_type": "old_value_test", "check_name": "previous_equals_subtracted_log_terms_expression_count", "passed": pd.NA, "affected_rows": int(old_vs_fresh["previous_equals_subtracted_log_terms_expression"].sum()), "details": "Rows where previous reconstructed value equals the requested subtractive log-term expression within 1e-10."},
    {"audit_type": "old_value_test", "check_name": "previous_equals_normalised_component_expression_count", "passed": pd.NA, "affected_rows": int(old_vs_fresh["previous_equals_normalised_component_log_expression"].sum()), "details": "Rows where previous reconstructed value equals a max-normalised component expression within 1e-10."},
]

dfsi_expression_audit = pd.DataFrame(assertion_records + metric_records)
display(dfsi_expression_audit)
dfsi_expression_audit.to_csv(TABLE_DIR / "dfsi_expression_audit.csv", index=False)

,audit_type,check_name,passed,affected_rows,details
0,assertion,all_complete_row_raw_components_are_nonnegative,True,0,"Checks E, D, F, I, A, and P after independent float64 conversion."
1,assertion,population_is_greater_than_zero,True,0,Required because term_population = log10(P).
2,assertion,every_complete_row_log_term_is_nonnegative,True,0,Each term column is expected to be nonnegative for nonnegative components and P > 0; P >= 1 in these data.
3,assertion,product_fresh_is_greater_than_zero,False,709,This uses the requested raw-subtraction expression for product_fresh.
4,assertion,dfsi_fresh_is_greater_than_zero,False,709,This uses term_events minus all other log terms exactly as requested.
5,assertion,dfsi_fresh_and_dfsi_product_fresh_agree_within_1e_10,False,709,Comparable rows: 0; max absolute difference: nan.
6,metric,one_to_one_rows,<NA>,720,Rows after fresh raw-source one-to-one district-name join.
7,metric,complete_rows,<NA>,709,Rows with all six raw components present.
8,metric,dfsi_fresh_min,<NA>,<NA>,-15.581415
9,metric,dfsi_fresh_max,<NA>,<NA>,-4.40686


## Save Term Contributions

In [9]:
term_output_columns = [
    "district_review_key", "published_district", "State_Name", "Dist_Name_impact", "E", "D", "F", "I", "A", "P",
    "term_events", "term_duration", "term_fatalities", "term_injured", "term_flooded_area", "term_population",
    "dfsi_fresh", "product_fresh", "dfsi_product_fresh", "previous_reconstructed_dfsi", "published_dfsi",
    "raw_E_flood_events_missing", "lgd_codes_seen", "lgd_code_count",
]
term_contributions = term_contrib[term_output_columns].copy()
term_contributions.to_csv(TABLE_DIR / "dfsi_term_contributions.csv", index=False)
print(f"Saved term contribution rows: {len(term_contributions):,}")

Saved term contribution rows: 720


## Named District Checks

In [10]:
def row_label(row):
    return f"{row['published_district']}, {row['State_Name']}"

complete_for_selection = term_contrib.loc[complete_row_mask].copy()
selected_frames = []

explicit_targets = [
    ("Patna", "BIHAR", "explicit_named_district"),
    ("Murshidabad", "WEST BENGAL", "explicit_named_district"),
    ("Thane", "MAHARASHTRA", "explicit_named_district"),
]
for district, state, reason in explicit_targets:
    mask = term_contrib["published_district"].map(make_review_key).eq(make_review_key(district)) & term_contrib["State_Name"].map(make_review_key).eq(make_review_key(state))
    selected = term_contrib.loc[mask].copy()
    selected["selection_reason"] = reason
    selected_frames.append(selected)

min_published = term_contrib.loc[term_contrib["published_dfsi"].idxmin()].to_frame().T.copy()
min_published["selection_reason"] = "minimum_published_dfsi_in_one_to_one_table"
selected_frames.append(min_published)

if term_contrib["previous_reconstructed_dfsi"].notna().any():
    max_previous = term_contrib.loc[term_contrib["previous_reconstructed_dfsi"].idxmax()].to_frame().T.copy()
    max_previous["selection_reason"] = "maximum_previous_reconstructed_dfsi"
    selected_frames.append(max_previous)

named_checks = pd.concat(selected_frames, ignore_index=True)
named_checks["arithmetic"] = named_checks.apply(
    lambda row: (
        f"log10(1 + E) - log10(1 + D) - 2*log10(1 + F) - log10(1 + I) - log10(1 + A) - log10(P) "
        f"= {row['term_events']:.12f} - {row['term_duration']:.12f} - {row['term_fatalities']:.12f} - "
        f"{row['term_injured']:.12f} - {row['term_flooded_area']:.12f} - {row['term_population']:.12f} = {row['dfsi_fresh']:.12f}"
    ),
    axis=1,
)

named_output_columns = [
    "selection_reason", "published_district", "State_Name", "E", "D", "F", "I", "A", "P",
    "term_events", "term_duration", "term_fatalities", "term_injured", "term_flooded_area", "term_population",
    "dfsi_fresh", "dfsi_product_fresh", "previous_reconstructed_dfsi", "published_dfsi", "arithmetic",
]
named_district_checks = named_checks[named_output_columns].copy()
display(named_district_checks)
named_district_checks.to_csv(TABLE_DIR / "dfsi_named_district_checks.csv", index=False)

,selection_reason,published_district,State_Name,E,D,F,I,A,P,term_events,term_duration,term_fatalities,term_injured,term_flooded_area,term_population,dfsi_fresh,dfsi_product_fresh,previous_reconstructed_dfsi,published_dfsi,arithmetic
0,explicit_named_district,Patna,BIHAR,71.0,16.0,381.0,30.0,18.820619,6868950.0,1.857332,1.230449,5.164127,1.491362,1.297117,6.83689,-14.162612,NaN,-14.162612,19.300564,log10(1 + E) - log10(1 + D) - 2*log10(1 + F) - log10(1 + I) - log10(1 + A) - log10(P) = 1.857332496431 - 1.230448921378 - 5.164126725823 - 1.491361693834 - 1.297117222188 - 6.836890355101 = -14.162612421893
1,explicit_named_district,Murshidabad,WEST BENGAL,44.0,23.0,435.0,9.0,10.404534,8350690.0,1.653213,1.380211,5.278973,1.0,1.057078,6.921722,-13.984772,NaN,-13.984772,18.910884,log10(1 + E) - log10(1 + D) - 2*log10(1 + F) - log10(1 + I) - log10(1 + A) - log10(P) = 1.653212513775 - 1.380211241712 - 5.278972978537 - 1.000000000000 - 1.057077555337 - 6.921722361808 = -13.984771623619
2,explicit_named_district,Thane,MAHARASHTRA,97.0,3.0,418.0,204.0,1.253628,9950177.0,1.991226,0.60206,5.244428,2.311754,0.352882,6.997831,-13.517729,NaN,-13.517729,18.861789,log10(1 + E) - log10(1 + D) - 2*log10(1 + F) - log10(1 + I) - log10(1 + A) - log10(P) = 1.991226075692 - 0.602059991328 - 5.244428045933 - 2.311753861056 - 0.352882187661 - 6.997830806318 = -13.517728816602
3,minimum_published_dfsi_in_one_to_one_table,Mahe,PUDUCHERRY,1.0,7.0,0.0,0.0,0.243893,50846.0,0.30103,0.90309,0.0,0.0,0.094783,4.706257,-5.4031,NaN,-5.4031,4.244485,log10(1 + E) - log10(1 + D) - 2*log10(1 + F) - log10(1 + I) - log10(1 + A) - log10(P) = 0.301029995664 - 0.903089986992 - 0.000000000000 - 0.000000000000 - 0.094783093316 - 4.706256793124 = -5.403099877768
4,maximum_previous_reconstructed_dfsi,East Kameng,ARUNACHAL PRADESH,23.0,2.0,0.0,0.0,1.326925,87734.0,1.380211,0.477121,0.0,0.0,0.366782,4.943168,-4.40686,NaN,-4.40686,7.516649,log10(1 + E) - log10(1 + D) - 2*log10(1 + F) - log10(1 + I) - log10(1 + A) - log10(P) = 1.380211241712 - 0.477121254720 - 0.000000000000 - 0.000000000000 - 0.366782446260 - 4.943167930320 = -4.406860389588


## Required Answers

In [ ]:
    source_hits = dfsi_expression_source_scan.copy()
    subtraction_hits = source_hits[source_hits["pattern_name"].isin(["-np.log10", "subtraction_between_log_terms", "assignments_to_reconstructed_dfsi", "overwriting_of_calculated_dfsi"])]
    previous_equals_subtractive_count = int(old_vs_fresh["previous_equals_subtracted_log_terms_expression"].sum())
    previous_equals_negative_count = int(old_vs_fresh["previous_equals_negative_dfsi_fresh"].sum())

    answer_rows = pd.DataFrame([
        {
            "question": "A. Which exact line or transformation created the negative values?",
            "answer": "The negative values are created by the subtractive log-term assignment itself: dfsi_fresh = term_events - term_duration - term_fatalities - term_injured - term_flooded_area - term_population. The same pattern appears in previous notebooks as chained '- np.log10(...)' terms assigned to reconstructed_dfsi/dfsi_calculated.",
        },
        {
            "question": "B. Was the previous reconstructed result approximately the negative of the correct calculation?",
            "answer": f"No. previous_equals_negative_dfsi_fresh matched {previous_equals_negative_count} rows; previous_equals_subtracted_log_terms_expression matched {previous_equals_subtractive_count} rows.",
        },
        {
            "question": "C. What is the fresh reconstructed range?",
            "answer": f"{complete['dfsi_fresh'].min()} to {complete['dfsi_fresh'].max()} over {len(complete)} complete rows.",
        },
        {
            "question": "D. What is the fresh Spearman correlation?",
            "answer": str(spearman),
        },
        {
            "question": "E. What discrepancies remain after correcting the expression?",
            "answer": "The requested expression audit does not correct the sign issue: dfsi_fresh remains negative for every complete row, product_fresh from the requested raw-subtraction expression is never positive, dfsi_product_fresh is undefined, and the values remain far outside the published positive DFSI range. The paper/source expression must be reconciled before sensitivity analysis.",
        },
    ])
    display(answer_rows)

    # Append answers to expression audit for a single audit table containing final conclusions.
    answer_audit_rows = answer_rows.rename(columns={"question": "check_name", "answer": "details"})
    answer_audit_rows.insert(0, "audit_type", "answer")
    answer_audit_rows.insert(2, "passed", pd.NA)
    answer_audit_rows.insert(3, "affected_rows", pd.NA)
    combined_expression_audit = pd.concat([dfsi_expression_audit, answer_audit_rows[dfsi_expression_audit.columns]], ignore_index=True)
    combined_expression_audit.to_csv(TABLE_DIR / "dfsi_expression_audit.csv", index=False)
    print("Updated dfsi_expression_audit.csv with final answers.")

,question,answer
0,A. Which exact line or transformation created the negative values?,The negative values are created by the subtractive log-term assignment itself: dfsi_fresh = term_events - term_duration - term_fatalities - term_injured - term_flooded_area - term_population. The same pattern appears...
1,B. Was the previous reconstructed result approximately the negative of the correct calculation?,No. previous_equals_negative_dfsi_fresh matched 0 rows; previous_equals_subtracted_log_terms_expression matched 709 rows.
2,C. What is the fresh reconstructed range?,-15.581414641823324 to -4.406860389587985 over 709 complete rows.
3,D. What is the fresh Spearman correlation?,-0.9344741971467349
4,E. What discrepancies remain after correcting the expression?,"The requested expression audit does not correct the sign issue: dfsi_fresh remains negative for every complete row, product_fresh from the requested raw-subtraction expression is never positive, dfsi_product_fresh is..."


Updated dfsi_expression_audit.csv with final answers.
